In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 15:18:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../../data/25.06/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet("../../../data/intermediate_files/l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)

g_p_s = session.spark.read.parquet("../../../data/intermediate_files/genes_therapeutic_areas")
g_p_s.count()

26/02/25 15:18:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

8285

In [4]:
filtered_data = (
    g_p_s.filter(f.col("uniqueTherapeuticAreas") < 6)
    .filter(f.col("uniqueTherapeuticAreas") > 1)
    .select("geneId")
    .distinct()
)

evidence = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full.filter(f.col("VEP") == 1)
    .join(filtered_data, on="geneId", how="inner")
    .drop("diseaseIds"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)

enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.1,
    efo_ancestors_to_remove=["MONDO_0045024"],
)

In [5]:
l2g_full_expl = l2g_full.withColumn("diseaseId", f.explode(f.col("diseaseIds"))).cache()
l2g_full_expl.count()

77071

In [6]:
l2g_best_cat = l2g_full_expl.filter(f.col("VEP") == 1).join(filtered_data, on="geneId", how="inner")

In [7]:
l2g_best_cat.count()

4742

In [8]:
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,5.540636,2.413890e-04,1.750932,17.532742,1.156574,1.111260,1.203735,9.789271e-13,6160,31130,3,84,12121
1,3+,5.897585,7.299695e-13,3.379999,10.290391,1.844411,1.674626,2.031410,1.925004e-35,20558,16732,15,72,12121
2,4+,10.288962,8.072448e-25,6.707865,15.781881,4.843708,4.051253,5.791173,4.052576e-67,32777,4513,36,51,12121


In [ ]:
l2g_best_cat.select("geneId", "diseaseId").distinct().count()

2734

In [ ]:
path_to_intermediate_data_folder = "../../../data/intermediate_files/"

In [ ]:
l2g_best_cat.toPandas().to_csv(path_to_intermediate_data_folder + "best_category.csv", sep="\t")

# Distirbution by TA


In [6]:
unique_tas = (
    disease_index_orig.select(f.explode(f.col("therapeuticAreas")).alias("ta"))
    .distinct()
    .rdd.map(lambda x: x[0])
    .collect()
)
for ta in sorted(unique_tas):
    print(ta)

EFO_0000319
EFO_0000540
EFO_0000618
EFO_0000651
EFO_0001379
EFO_0001444
EFO_0002571
EFO_0005741
EFO_0005803
EFO_0005932
EFO_0009605
EFO_0009690
EFO_0010282
EFO_0010285
GO_0008150
MONDO_0002025
MONDO_0021205
MONDO_0024458
MONDO_0045024
OTAR_0000006
OTAR_0000009
OTAR_0000010
OTAR_0000014
OTAR_0000017
OTAR_0000018
OTAR_0000020


In [7]:
ta_to_exclude = ["EFO_0001444", "MONDO_0045024", "GO_0008150", "EFO_0000651"]

In [8]:
ta_to_anlyze = [ta for ta in unique_tas if ta not in ta_to_exclude]

In [ ]:
names_ta = disease_index_orig.filter(f.col("id").isin(unique_tas)).select("id", "name").toPandas()

In [ ]:
import pandas as pd

results = []

for ta in ta_to_anlyze:
    efo_to_include = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
        disease_index_orig=disease_index_orig,
        efo_ids=ta,
    )
    df = l2g_full_expl.filter(f.col("diseaseId").isin(efo_to_include))
    df = df.select("geneId", "diseaseId").distinct()

    N = df.count()
    unique_diseases = df.select("diseaseId").distinct().count()
    unique_targets = df.select("geneId").distinct().count()

    results.append(
        {"therapeutic_area": ta, "N": N, "unique_diseases": unique_diseases, "unique_targets": unique_targets}
    )

# Convert to pandas DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df = results_df.merge(names_ta[["id", "name"]], left_on="therapeutic_area", right_on="id", how="left").drop(
    columns=["id"]
)

In [ ]:
print(results_df)

   therapeutic_area     N  unique_diseases  unique_targets  \
0       EFO_0005932     0                0               0   
1       EFO_0005803   891               55             594   
2       EFO_0010285  3063              109            1419   
3      OTAR_0000020  3673               51            2303   
4     MONDO_0002025  3703               75            1774   
5       EFO_0005741   589               57             465   
6       EFO_0001379  3987              104            2277   
7      OTAR_0000014   155               11             115   
8      OTAR_0000009   255               39             162   
9      OTAR_0000010  2196               76            1113   
10    MONDO_0021205   133               14             114   
11      EFO_0009690  1257               64             854   
12      EFO_0010282  3354              146            1619   
13      EFO_0002571   725               24             492   
14      EFO_0000618  5686              200            2770   
15    MO

In [ ]:
import pandas as pd

results = []

for ta in ta_to_anlyze:
    efo_to_include = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
        disease_index_orig=disease_index_orig,
        efo_ids=ta,
    )
    df = l2g_best_cat.filter(f.col("diseaseId").isin(efo_to_include))
    df = df.select("geneId", "diseaseId").distinct()

    N = df.count()
    unique_diseases = df.select("diseaseId").distinct().count()
    unique_targets = df.select("geneId").distinct().count()

    results.append(
        {"therapeutic_area": ta, "N": N, "unique_diseases": unique_diseases, "unique_targets": unique_targets}
    )

# Convert to pandas DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df = results_df.merge(names_ta[["id", "name"]], left_on="therapeutic_area", right_on="id", how="left").drop(
    columns=["id"]
)

In [ ]:
print(results_df)

   therapeutic_area    N  unique_diseases  unique_targets  \
0       EFO_0005932    0                0               0   
1       EFO_0005803   76               21              56   
2       EFO_0010285  233               51             119   
3      OTAR_0000020  272               32             192   
4     MONDO_0002025  266               44             143   
5       EFO_0005741   28               14              23   
6       EFO_0001379  285               46             187   
7      OTAR_0000014    9                5               8   
8      OTAR_0000009   18                9              14   
9      OTAR_0000010  144               37              97   
10    MONDO_0021205   21                7              16   
11      EFO_0009690   95               26              76   
12      EFO_0010282  206               60             119   
13      EFO_0002571   65               11              58   
14      EFO_0000618  433               92             248   
15    MONDO_0024458  138

# By target class


In [26]:
target = session.spark.read.parquet(path_to_release_folder + "output/target")

In [ ]:
l1_targets = (
    target.select(f.explode(f.col("targetClass")).alias("tc"))
    .filter(f.col("tc.level").contains("l1"))
    .select("tc.id", "tc.label", "tc.level")
    .distinct()
    .orderBy("level")
)

In [28]:
l1_classes = l1_targets.select("label").distinct().toPandas()
print(l1_classes)

                          label
0                        Enzyme
1         Other nuclear protein
2                      Adhesion
3               Surface antigen
4             Membrane receptor
5          Epigenetic regulator
6       Other cytosolic protein
7        Other membrane protein
8                   Ion channel
9   Auxiliary transport protein
10         Transcription factor
11         Unclassified protein
12                  Transporter
13             Secreted protein
14           Structural protein


In [29]:
l1_classes = l1_classes["label"].tolist()

In [30]:
import pandas as pd

results = []

for tid in l1_classes:
    target_to_use = (
        target.filter(f.array_contains(f.col("targetClass.label"), tid)).select("id").withColumnRenamed("id", "geneId")
    )
    df = l2g_full_expl.join(target_to_use, "geneId", "inner")
    df = df.select("geneId", "diseaseId").distinct()

    N = df.count()
    unique_diseases = df.select("diseaseId").distinct().count()
    unique_targets = df.select("geneId").distinct().count()

    results.append({"l1_class": tid, "N": N, "unique_diseases": unique_diseases, "unique_targets": unique_targets})

# Convert to pandas DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df

,l1_class,N,unique_diseases,unique_targets
0,Enzyme,5857,835,1151
1,Other nuclear protein,94,79,20
2,Adhesion,208,163,29
3,Surface antigen,62,49,15
4,Membrane receptor,1156,398,235
5,Epigenetic regulator,420,196,99
6,Other cytosolic protein,399,209,65
7,Other membrane protein,29,28,9
8,Ion channel,626,264,133
9,Auxiliary transport protein,92,67,17


In [32]:
import pandas as pd

results = []

for tid in l1_classes:
    target_to_use = (
        target.filter(f.array_contains(f.col("targetClass.label"), tid)).select("id").withColumnRenamed("id", "geneId")
    )
    df = l2g_best_cat.join(target_to_use, "geneId", "inner")
    df = df.select("geneId", "diseaseId").distinct()

    N = df.count()
    unique_diseases = df.select("diseaseId").distinct().count()
    unique_targets = df.select("geneId").distinct().count()

    results.append({"l1_class": tid, "N": N, "unique_diseases": unique_diseases, "unique_targets": unique_targets})

# Convert to pandas DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df

,l1_class,N,unique_diseases,unique_targets
0,Enzyme,441,233,148
1,Other nuclear protein,5,5,2
2,Adhesion,25,21,7
3,Surface antigen,0,0,0
4,Membrane receptor,136,93,37
5,Epigenetic regulator,23,20,8
6,Other cytosolic protein,10,10,6
7,Other membrane protein,1,1,1
8,Ion channel,42,33,15
9,Auxiliary transport protein,0,0,0
